### Chains Branching - Practice & Exercises

In this notebook, you will practice routing execution flows dynamically based on criteria using `RunnableBranch`.

#### Part 1: Follow-Along Practice

##### Sentiment Feedback Router

##### Goal / Question
Classify the sentiment of user feedback (positive, negative, neutral, or escalate) and dynamically route execution to the appropriate support responder chain.

##### Sample Input
```python
{"feedback": "The product is terrible. It broke after just one use and the quality is very poor."}
```

##### Sample Output
An automated customer service response addressing the negative sentiment of the feedback.

##### Plan / Steps
1. Create specialized prompt templates for positive, negative, neutral, and escalation feedback.
2. Construct a classification chain that analyzes feedback and returns a category string ("positive", "negative", "neutral", "escalate").
3. Set up a `RunnableBranch` that contains conditional checks mapped to each responder chain, plus a default fallback.
4. Compose the final pipeline by connecting the classification chain to the branching router: `chain = classification_chain | branches`.
5. Invoke the pipeline with user feedback.

In [ ]:
from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableBranch
from langchain_openai import ChatOpenAI

load_dotenv()
model = ChatOpenAI(model="gpt-4o")

positive_feedback_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a thank you note for this positive feedback: {feedback}."),
])
positive_chain = positive_feedback_template | model | StrOutputParser()

negative_feedback_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a response addressing this negative feedback: {feedback}."),
])
negative_chain = negative_feedback_template | model | StrOutputParser()

neutral_feedback_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a request for more details for this neutral feedback: {feedback}."),
])
neutral_chain = neutral_feedback_template | model | StrOutputParser()

escalate_feedback_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a message to escalate this feedback to a human agent: {feedback}."),
])
escalate_chain = escalate_feedback_template | model | StrOutputParser()

classification_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Classify the sentiment of this feedback as positive, negative, neutral, or escalate: {feedback}."),
])
classification_chain = classification_template | model | StrOutputParser()

branches = RunnableBranch(
    (lambda x: "positive" in x, positive_chain),
    (lambda x: "negative" in x, negative_chain),
    (lambda x: "neutral" in x, neutral_chain),
    escalate_chain
)

chain = classification_chain | branches
review = "The product is terrible. It broke after just one use and the quality is very poor."
result = chain.invoke({"feedback": review})
print(result)

#### Part 2: Practice Exercises

##### Exercise: Multi-Subject Router

##### Goal / Question
Write a router chain that classifies an incoming question as "math", "coding", or "general", and dynamically executes the corresponding dedicated solver chain.

##### Sample Input
```python
{"question": "What is the derivative of x^2?"}
```

##### Sample Output
The mathematical step-by-step derivative solution: 2x.

##### Plan / Steps
1. Define specialized prompt templates for math solvers, coding generation, and general queries.
2. Link each template to a chain with model and parser.
3. Create a classifier chain `router_chain` prompting the model to classify questions as math, coding, or general.
4. Configure a `RunnableBranch` that inspects the classification string (e.g. `"math" in x.lower()`) and routes the prompt input dictionary to the target chain, using the general chain as fallback.
5. Pipe the classifier and router together: `full_router_chain = router_chain | subject_branches`.
6. Invoke the router.

In [ ]:
# TODO: Write your implementation here

<details>
<summary>💡 Click to see solution</summary>

```python
math_prompt = ChatPromptTemplate.from_template("Solve this math problem: {question}")
coding_prompt = ChatPromptTemplate.from_template("Write python code for: {question}")
general_prompt = ChatPromptTemplate.from_template("Answer this general question: {question}")

math_chain = math_prompt | model | StrOutputParser()
coding_chain = coding_prompt | model | StrOutputParser()
general_chain = general_prompt | model | StrOutputParser()

router_prompt = ChatPromptTemplate.from_template("Classify this question as math, coding, or general: {question}")
router_chain = router_prompt | model | StrOutputParser()

subject_branches = RunnableBranch(
    (lambda x: "math" in x.lower(), math_chain),
    (lambda x: "coding" in x.lower(), coding_chain),
    general_chain
)

full_router_chain = router_chain | subject_branches
print(full_router_chain.invoke({"question": "What is the derivative of x^2?"}))
```
</details>